# Total Part Risk Score — v5
### Fixes the structurally-empty High tier found during a full audit of the v4 methodology

## 0. The bug, proven with numbers before anything else

**Claim under investigation:** the part-level recommendation engine never produces a High-tier
part. **Root-caused, not assumed:** this is not a Step 6 (recommendation engine) bug at all -- it
is a scale-calibration defect in the Step 4 scoring formula itself (`TPRS = L x I / 100` with
20/40/60/80 fixed thresholds).

Checked directly against all 433 real parts:

```
Global max of L x I / 100 across every real part, hard-stop or not:  56.1
Number of parts (out of 433) that reach 60+ from the raw formula:     0
Number of parts that reach 80+ from the raw formula:                  0
```

**Every single part in this dataset, without exception, tops out at 56.1** on the raw formula.
The entire Critical tier that existed in v4 (31 parts) was populated *only* by the hard-stop
override (a categorical rule forcing `TPRS = 100`), never by the continuous formula reaching that
range on its own. High and Critical were both structurally unreachable through the formula itself.

**Why, mechanically:** `L = 0.667*N_Compliance + 0.333*N_Manufacturer`. Real `N_Manufacturer`
values are almost always near zero (median 4.8 out of 100 across all 433 parts -- confirmed
repeatedly in this project's audit history). Even with `N_Compliance` maxed at 100, `L` tops out
around 68-81 depending on the rare high-`N_Manufacturer` case. `I` similarly cannot realistically
reach 100. The product of two sub-100 numbers, divided by 100, structurally cannot reach the 60-79
and 80-100 zones the original thresholds assumed were reachable.

## 1. The fix: min-max stretch, not new thresholds

The **same technique already used elsewhere in this project** (tier-anchored normalization of the
individual N_ factors) applies here: a formula's raw output should be scaled to the range it can
actually achieve before comparing it against fixed percentage-of-100 thresholds, or those
thresholds are being compared against a range the data can never fill.

$$
TPRS_{stretched} = \frac{TPRS_{raw} - \min(TPRS_{raw})}{\max(TPRS_{raw}) - \min(TPRS_{raw})} \times 100
$$

This is a **linear** rescaling (preserves relative ordering and relative distances between parts
exactly -- it is not a rank/quantile transform, which would distort those distances). The
same 20/40/60/80 thresholds are then applied to the *stretched* score, and now mean what they were
always supposed to mean: position within the realistically achievable range, not position within
an assumed-but-unreachable 0-100 range.

**The hard-stop override is unchanged** -- it remains a separate, categorical business rule
(Compliance = High Risk AND Alternative = Critical -> forced Critical), applied *after* the
stretch, exactly as in v4. This fix does not touch that logic at all.

**Honest disclosure:** this rescaling means `raw_max` (56.1 in this dataset) will shift if a
future part scores worse than any part seen so far -- the stretch is *relative to the population
in this file*, not an absolute physical ceiling. That is disclosed here rather than hidden, and is
the same caveat that applies to the earlier tier-anchored N_ factor normalization elsewhere in
this project. A production system ingesting new parts over time should periodically re-check
whether `raw_max` has moved and re-stretch if so -- this is flagged explicitly in Section 4.


In [4]:
# CELL 1: Load and rebuild the base scoring pipeline (same source data as v4)
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', 25)
pd.set_option('display.width', 160)

df = pd.read_csv('total_part_risk_score_v4.csv')
print(f"Loaded: {df.shape}")
print(f"Sub-risk columns present: {[c for c in df.columns if 'risk' in c.lower()]}")


Loaded: (433, 20)
Sub-risk columns present: ['stock_risk_score', 'stock_risk_category', 'manufacturer_risk_score', 'manufacturer_risk_level', 'alternative_risk_score', 'alternative_risk_level', 'compliance_risk_score', 'compliance_risk_level', 'inventory_risk_score', 'inventory_risk_level']


## 2. Normalization of the five sub-risk scores (unchanged from v4 -- audited, not modified)


In [5]:
# CELL 2: Tier-anchored normalization (verbatim from v4 -- audited during this rebuild, no defect found here)

TIER_ORDERS = {
    'stock_risk_category':     (['Low','Medium','High'], 15),
    'manufacturer_risk_level': (['Low','Medium','High','Critical'], 10),
    'alternative_risk_level':  (['Very Low','Low','Medium','High','Critical'], 8),
    'compliance_risk_level':   (['Low Risk','Medium Risk','High Risk'], 15),
}

def normalize_factor(sub_df, level_col, score_col, out_col):
    tiers, window = TIER_ORDERS[level_col]
    n = len(tiers)
    anchors = {t: round(100 * i / (n - 1), 2) for i, t in enumerate(tiers)}
    band_stats = sub_df.groupby(level_col)[score_col].agg(['min', 'max'])

    def compute(row):
        lvl, score = row[level_col], row[score_col]
        if pd.isna(lvl) or pd.isna(score):
            return np.nan
        anchor = anchors[lvl]
        bmin, bmax = band_stats.loc[lvl, 'min'], band_stats.loc[lvl, 'max']
        pos = 0.5 if bmax == bmin else (score - bmin) / (bmax - bmin)
        tier_idx = tiers.index(lvl)
        if tier_idx == 0:
            refined = anchor + pos * window
        elif tier_idx == n - 1:
            refined = anchor - (1 - pos) * window
        else:
            refined = anchor + (pos - 0.5) * 2 * window
        return float(np.clip(refined, 0, 100))

    sub_df[out_col] = sub_df.apply(compute, axis=1)
    return sub_df

norm = df.copy()
norm = normalize_factor(norm, 'stock_risk_category',     'stock_risk_score',        'N_STOCK')
norm = normalize_factor(norm, 'manufacturer_risk_level',  'manufacturer_risk_score', 'N_MFR')
norm = normalize_factor(norm, 'alternative_risk_level',   'alternative_risk_score',  'N_ALT')
norm = normalize_factor(norm, 'compliance_risk_level',    'compliance_risk_score',   'N_COMP')

for c in ['N_STOCK','N_MFR','N_ALT','N_COMP']:
    assert norm[c].between(0, 100).all(), f"{c} out of bounds -- normalization defect"
print("All four normalized factors confirmed within [0, 100].")
norm[['N_STOCK','N_MFR','N_ALT','N_COMP']].describe()


All four normalized factors confirmed within [0, 100].


,N_STOCK,N_MFR,N_ALT,N_COMP
count,433.000000,433.000000,433.000000,433.000000
mean,37.635014,6.770094,34.240185,30.323326
std,34.929971,7.286228,37.646289,34.033607
min,0.000000,0.000000,0.000000,7.500000
25%,6.324366,4.783542,8.000000,7.500000
50%,13.246249,4.783542,8.000000,7.500000
75%,85.866834,4.783542,75.000000,65.000000
max,100.000000,43.330000,96.000000,100.000000


## 3. AHP weights (unchanged from v4 -- audited, Consistency Ratios re-verified)


In [6]:
# CELL 3: AHP weights, re-verified

def ahp_solve(M, names):
    n = len(names)
    col_norm = M / M.sum(axis=0)
    w = col_norm.mean(axis=1)
    Aw = M @ w
    lam_max = np.mean(Aw / w)
    CI = (lam_max - n) / (n - 1) if n > 2 else 0.0
    RI_TABLE = {1: 0, 2: 0, 3: 0.58, 4: 0.90, 5: 1.12, 6: 1.24}
    CR = CI / RI_TABLE[n] if RI_TABLE[n] > 0 else 0.0
    return dict(zip(names, w)), CR

names_L = ['Compliance', 'Manufacturer']
M_L = np.array([[1, 2], [1/2, 1]])
w_L, cr_L = ahp_solve(M_L, names_L)

names_I = ['Alternative', 'Stock']
M_I = np.array([[1, 3], [1/3, 1]])
w_I, cr_I = ahp_solve(M_I, names_I)

print("Likelihood weights:", {k: round(v, 3) for k, v in w_L.items()}, f"CR={cr_L:.4f}")
print("Impact weights:     ", {k: round(v, 3) for k, v in w_I.items()}, f"CR={cr_I:.4f}")
assert cr_L < 0.10 and cr_I < 0.10, "AHP consistency check failed"


Likelihood weights: {'Compliance': 0.667, 'Manufacturer': 0.333} CR=0.0000
Impact weights:      {'Alternative': 0.75, 'Stock': 0.25} CR=0.0000


## 4. The composite score — L x I, min-max stretch, hard-stop override

This is the section that actually changes from v4. Everything above this point was audited and
found correct; the defect was isolated entirely to how the raw composite gets classified.


In [7]:
# CELL 4: Composite score with the stretch fix

norm['L'] = w_L['Compliance'] * norm['N_COMP'] + w_L['Manufacturer'] * norm['N_MFR']
norm['I'] = w_I['Alternative'] * norm['N_ALT'] + w_I['Stock'] * norm['N_STOCK']
norm['TPRS_raw'] = np.clip(norm['L'] * norm['I'] / 100, 0, 100)

# --- THE FIX ---
_raw_min, _raw_max = norm['TPRS_raw'].min(), norm['TPRS_raw'].max()
print(f"Realistic achievable range this run: {_raw_min:.3f} to {_raw_max:.3f}")
print(f"(Compare to v4's assumption that this range spanned 0 to 100 -- it never has, for any real part)")

STRETCH_MIN = _raw_min   # persisted below so future re-runs can detect drift (Section 0's honest disclosure)
STRETCH_MAX = _raw_max

norm['TPRS_stretched'] = (norm['TPRS_raw'] - STRETCH_MIN) / (STRETCH_MAX - STRETCH_MIN) * 100

norm['hard_stop'] = (norm['compliance_risk_level'] == 'High Risk') & (norm['alternative_risk_level'] == 'Critical')
norm['TPRS'] = np.where(norm['hard_stop'], 100.0, norm['TPRS_stretched']).round(1)

def classify(score):
    if score < 20: return 'Very Low'
    if score < 40: return 'Low'
    if score < 60: return 'Medium'
    if score < 80: return 'High'
    return 'Critical'

norm['TPRS_category'] = norm['TPRS'].apply(classify)

print()
print("Category distribution -- ALL FIVE TIERS NOW POPULATED:")
print(norm['TPRS_category'].value_counts().reindex(['Very Low','Low','Medium','High','Critical']))


Realistic achievable range this run: 0.012 to 56.080
(Compare to v4's assumption that this range spanned 0 to 100 -- it never has, for any real part)

Category distribution -- ALL FIVE TIERS NOW POPULATED:
TPRS_category
Very Low    339
Low          34
Medium        6
High         19
Critical     35
Name: count, dtype: int64


In [8]:
# CELL 5: Validation -- prove the fix, don't just assert it worked

assert set(norm['TPRS_category'].unique()) == {'Very Low','Low','Medium','High','Critical'}, \
    "FIX FAILED: not all five tiers are populated"
assert (norm['TPRS_category']=='High').sum() > 0, "FIX FAILED: High tier still empty"
print(f"High tier populated: {(norm['TPRS_category']=='High').sum()} parts")

# Hard-stop parts must always be Critical (business rule preserved, unaffected by the stretch)
hs = norm[norm['hard_stop']]
assert (hs['TPRS_category'] == 'Critical').all(), "Hard-stop override broken by the stretch fix"
print(f"All {len(hs)} hard-stop parts confirmed Critical (override unaffected by the fix).")

# Monotonicity: stretching must preserve rank order exactly (linear transform property)
_rank_before = norm['TPRS_raw'].rank()
_rank_after = norm['TPRS_stretched'].rank()
assert (_rank_before == _rank_after).all(), "Stretch changed relative ordering -- should be impossible for a linear transform"
print("Confirmed: relative ordering between all 433 parts is exactly preserved (linear transform property).")


High tier populated: 19 parts
All 31 hard-stop parts confirmed Critical (override unaffected by the fix).
Confirmed: relative ordering between all 433 parts is exactly preserved (linear transform property).


## 5. Edge case and boundary tests


In [9]:
# CELL 6: Boundary and edge-case tests

def stretch_and_classify(raw_value, raw_min=STRETCH_MIN, raw_max=STRETCH_MAX):
    stretched = (raw_value - raw_min) / (raw_max - raw_min) * 100
    return classify(np.clip(stretched, 0, 100))

# Boundary values exactly at each threshold
assert classify(19.999) == 'Very Low'
assert classify(20.0) == 'Low'
assert classify(39.999) == 'Low'
assert classify(40.0) == 'Medium'
assert classify(59.999) == 'Medium'
assert classify(60.0) == 'High'
assert classify(79.999) == 'High'
assert classify(80.0) == 'Critical'
assert classify(100.0) == 'Critical'
print("Threshold boundary tests passed (19.999/20/39.999/40/59.999/60/79.999/80/100).")

# Missing/NaN sub-scores must not silently produce a false-low score
_test_row = norm.iloc[0].copy()
_test_row['N_COMP'] = np.nan
_test_L = w_L['Compliance'] * _test_row['N_COMP'] + w_L['Manufacturer'] * _test_row['N_MFR']
assert pd.isna(_test_L), "A missing sub-score must propagate as NaN, not silently become 0 (which would understate risk)"
print("Missing-value propagation test passed: a NaN sub-score correctly produces a NaN composite, not a false zero.")

# Zero-risk floor: a hypothetically perfect part (all sub-scores at 0) must classify Very Low, not error
assert stretch_and_classify(0.0) == 'Very Low'
# Worst-case ceiling: a hypothetical part scoring above the current realistic max must still classify Critical
assert stretch_and_classify(STRETCH_MAX + 10) == 'Critical'
print("Zero-floor and above-ceiling edge cases both classify correctly.")


Threshold boundary tests passed (19.999/20/39.999/40/59.999/60/79.999/80/100).
Missing-value propagation test passed: a NaN sub-score correctly produces a NaN composite, not a false zero.
Zero-floor and above-ceiling edge cases both classify correctly.


## 6. Manual verification on sample parts across all five tiers


In [10]:
# CELL 7: Manual spot-check -- one real part per tier, full factor breakdown shown

for cat in ['Very Low','Low','Medium','High','Critical']:
    subset = norm[norm['TPRS_category']==cat]
    if len(subset):
        row = subset.iloc[len(subset)//2]
        print(f"--- {cat}: {row['mpn']} ---")
        print(f"  N_COMP={row['N_COMP']:.1f} N_MFR={row['N_MFR']:.1f} -> L={row['L']:.1f}")
        print(f"  N_ALT={row['N_ALT']:.1f} N_STOCK={row['N_STOCK']:.1f} -> I={row['I']:.1f}")
        print(f"  TPRS_raw={row['TPRS_raw']:.2f} -> stretched={row['TPRS_stretched']:.1f} -> "
              f"final={row['TPRS']:.1f} (hard_stop={row['hard_stop']})")
        print()


--- Very Low: ATTINY404SSNR ---
  N_COMP=7.5 N_MFR=4.8 -> L=6.6
  N_ALT=0.0 N_STOCK=100.0 -> I=25.0
  TPRS_raw=1.65 -> stretched=2.9 -> final=2.9 (hard_stop=False)

--- Low: GS12181INTE3 ---
  N_COMP=85.0 N_MFR=4.8 -> L=58.3
  N_ALT=8.0 N_STOCK=65.0 -> I=22.2
  TPRS_raw=12.96 -> stretched=23.1 -> final=23.1 (hard_stop=False)

--- Medium: NCP2990FCT2G ---
  N_COMP=65.0 N_MFR=4.8 -> L=44.9
  N_ALT=75.0 N_STOCK=8.3 -> I=58.3
  TPRS_raw=26.21 -> stretched=46.7 -> final=46.7 (hard_stop=False)

--- High: JLC1562BNG ---
  N_COMP=65.0 N_MFR=4.8 -> L=44.9
  N_ALT=75.0 N_STOCK=88.4 -> I=78.3
  TPRS_raw=35.20 -> stretched=62.8 -> final=62.8 (hard_stop=False)

--- Critical: LA7845E ---
  N_COMP=85.0 N_MFR=4.8 -> L=58.3
  N_ALT=96.0 N_STOCK=85.9 -> I=93.5
  TPRS_raw=54.45 -> stretched=97.1 -> final=100.0 (hard_stop=True)



## 7. Export


In [11]:
# CELL 8: Export

out_cols = ['mpn','stock_risk_score','stock_risk_category','N_STOCK',
            'manufacturer_risk_score','manufacturer_risk_level','N_MFR',
            'alternative_risk_score','alternative_risk_level','N_ALT',
            'compliance_risk_score','compliance_risk_level','N_COMP',
            'L','I','TPRS_raw','TPRS_stretched','TPRS','TPRS_category','hard_stop']
final = norm[out_cols].sort_values('TPRS', ascending=False)
final.to_csv('total_part_risk_score_v5.csv', index=False)
print(final.shape)
final['TPRS_category'].value_counts().reindex(['Very Low','Low','Medium','High','Critical'])


(433, 20)


TPRS_category
Very Low    339
Low          34
Medium        6
High         19
Critical     35
Name: count, dtype: int64

## 8. Summary

**Root cause:** the raw `L x I / 100` formula, given real-world sub-score distributions (especially
`N_Manufacturer` being almost always near zero), can never exceed ~56 for any part in this
dataset. The original fixed 20/40/60/80 thresholds assumed a 0-100 achievable range that never
existed in practice -- so High (60-79) and, without the hard-stop override, Critical (80-100) were
structurally unreachable.

**Fix:** a linear min-max stretch of the raw score to the range it can actually achieve, before
applying the same thresholds. This preserves every part's relative ranking exactly (proven in
Cell 5) and is the same normalization principle already used elsewhere in this project for the
individual sub-factors -- extended here to the composite score, which needed it just as much.

**Result:** all five tiers populated (Very Low, Low, Medium, High, Critical), hard-stop override
unaffected and re-verified, boundary/edge cases tested explicitly.

**Disclosed limitation:** the stretch is relative to the current 433-part population. If future
parts score worse than today's realistic max (56.1 raw), `STRETCH_MAX` should be recomputed and
the fleet re-classified -- this is a re-calibration step, not a one-time fix, and should be
scheduled periodically in production rather than assumed permanent.
